In [ ]:
from google.colab import files

# يطلب منك اختيار ورفع الملف يدوياً
uploaded = files.upload()


Saving kaggle (3).json to kaggle (3) (1).json


In [ ]:
import os
print(os.listdir())  # يجب أن ترى "kaggle.json" في القائمة


['.config', 'full_data.csv', 'kaggle (3).json', 'kaggle (3) (1).json', 'medal-emnlp.zip', 'results.csv', 'full_data.gz', 'chunks', 'sample_data']


In [ ]:
# إنشاء مجلد لكاجل وإعداد API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# التحقق من أن Kaggle API يعمل
!kaggle datasets list


cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 4, in <module>
    from kaggle.cli import main
  File "/usr/local/lib/python3.11/dist-packages/kaggle/__init__.py", line 7, in <module>
    api.authenticate()
  File "/usr/local/lib/python3.11/dist-packages/kaggle/api/kaggle_api_extended.py", line 407, in authenticate
    raise IOError('Could not find {}. Make sure it\'s located in'
OSError: Could not find kaggle.json. Make sure it's located in /root/.kaggle. Or use the environment method. See setup instructions at https://github.com/Kaggle/kaggle-api/


In [ ]:
# تحميل البيانات بدون حفظها محليًا
!kaggle datasets download -d xhlulu/medal-emnlp --unzip

Dataset URL: https://www.kaggle.com/datasets/xhlulu/medal-emnlp
License(s): other
medal-emnlp.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
import os

file_path = "full_data.csv"  # استبدله بمسار ملفك
file_size = os.path.getsize(file_path) / (1024 * 1024)  # تحويل الحجم إلى ميغابايت
print(f"حجم الملف: {file_size:.2f} ميغابايت")


حجم الملف: 567.38 ميغابايت


In [ ]:
# تحديد اسم ملف CSV غير المضغوط واسم الملف المضغوط (تأكد من الأسماء بعد التنزيل)
csv_file_path = "full_data.csv"   # ملف البيانات الرئيسي
zip_file_path = "medal-emnlp.zip"   # إذا كان هناك ملف مضغوط

In [ ]:
import zipfile

# اسم الملف المضغوط
zip_file_path = "medal-emnlp.zip"

# استخراج اسم ملف CSV داخل الأرشيف
with zipfile.ZipFile(zip_file_path, "r") as z:
    csv_file_path = z.namelist()[0]  # الحصول على اسم ملف CSV داخل الـ ZIP
    print(f"تم العثور على الملف داخل الأرشيف: {csv_file_path}")


تم العثور على الملف داخل الأرشيف: full_data.csv


In [ ]:
import psutil  # مكتبة لقياس استهلاك الذاكرة

def get_memory_usage():
    process = psutil.Process()
    mem_info = process.memory_info()
    return mem_info.rss / (1024 ** 2)  # تحويل إلى MB

# ✅ دالة لحساب استهلاك الذاكرة كنسبة مئوية
def get_memory_usage_percent():
    return psutil.virtual_memory().percent

In [ ]:
!pip install "dask[dataframe]"

# **Pandas**

In [ ]:
import pandas as pd
import time
import psutil
import gc
from tabulate import tabulate

# ✅ دوال قياس استهلاك الذاكرة
def get_memory_usage():
    return psutil.Process().memory_info().rss / (1024 * 1024)

def get_memory_usage_percent():
    return psutil.virtual_memory().percent

# ✅ تحميل البيانات باستخدام Pandas مع chunksize
csv_file_path = "full_data.csv"  # استبدل بمسار ملفك
chunk_size = 100000  # حجم الشانك
operation_type = f"Pandas (chunksize={chunk_size})"

mem_before = get_memory_usage()  # قياس الذاكرة قبل التنفيذ
start = time.time()

mem_during = 0  # متغير لحفظ أعلى استهلاك للذاكرة أثناء التشغيل

df_chunks = []  # قائمة لحفظ أجزاء البيانات
numeric_columns = None  # متغير لتخزين أسماء الأعمدة العددية فقط

with pd.read_csv(csv_file_path, chunksize=chunk_size) as chunks:
    for chunk in chunks:
        if numeric_columns is None:
            numeric_columns = chunk.select_dtypes(include=["number"]).columns  # تحديد الأعمدة العددية
        df_chunks.append(chunk[numeric_columns])  # الاحتفاظ فقط بالأعمدة العددية
        mem_during = max(mem_during, get_memory_usage_percent())  # تحديث أعلى استهلاك للذاكرة

# ✅ دمج جميع الأجزاء وتنفيذ عملية حسابية فعلية

df_final = pd.concat(df_chunks, ignore_index=True)
df_result = df_final.sum()  # تنفيذ عملية حسابية لإجبار التنفيذ

mem_after = get_memory_usage()  # قياس الذاكرة بعد التنفيذ
end = time.time()

# ✅ إنشاء DataFrame باللغة الإنجليزية
results_df = pd.DataFrame(columns=["Operation Type", "Time Taken (Seconds)", "Peak Memory Usage (%)", "Memory Increase (MB)"])
results_df.loc[0] = [operation_type, round(end - start, 2), round(mem_during, 2), round(mem_after - mem_before, 2)]

# ✅ حفظ النتائج في ملف CSV دون حذف البيانات السابقة
results_csv_path = "results.csv"
try:
    existing_results = pd.read_csv(results_csv_path)
    results_df = pd.concat([existing_results, results_df], ignore_index=True)
except FileNotFoundError:
    pass  # إذا لم يكن هناك ملف سابق، يتم إنشاؤه مباشرة

results_df.to_csv(results_csv_path, index=False, encoding="utf-8", sep=",")

# ✅ عرض النتائج في جدول منسق
print("✅ Results saved in results.csv\n")
print(tabulate(results_df, headers="keys", tablefmt="grid"))

# ✅ تحرير الذاكرة بعد انتهاء المعالجة
del df_chunks, df_final, df_result
gc.collect()


✅ Results saved in results.csv

+----+---------------------------+------------------------+-------------------------+------------------------+
|    | Operation Type            |   Time Taken (Seconds) |   Peak Memory Usage (%) |   Memory Increase (MB) |
+====+===========================+========================+=========================+========================+
|  0 | Pandas (chunksize=100000) |                  15.96 |                    12.7 |                 241.02 |
+----+---------------------------+------------------------+-------------------------+------------------------+


0

# **Dask**

In [ ]:
import pandas as pd
import dask.dataframe as dd
import time
import psutil
import gc
from tabulate import tabulate

# ✅ دوال قياس استهلاك الذاكرة
def get_memory_usage():
    return psutil.Process().memory_info().rss / (1024 * 1024)

def get_memory_usage_percent():
    return psutil.virtual_memory().percent

# ✅ تحديد نوع العملية
operation_type = "Dask (direct read)"

mem_before = get_memory_usage()  # قياس الذاكرة قبل التنفيذ
start = time.time()

mem_during = 0  # متغير لحفظ أعلى استهلاك للذاكرة أثناء التشغيل

# ✅ تحميل الملف باستخدام Dask
csv_file_path = "full_data.csv"  # استبدل بمسار ملفك
df_dask = dd.read_csv(csv_file_path)

# ✅ تحديد الأعمدة العددية فقط لتجنب الخطأ
numeric_columns = df_dask.select_dtypes(include=["number"]).columns
df_dask_numeric = df_dask[numeric_columns]

# ✅ تنفيذ عملية حسابية فعلية لإجبار التنفيذ (مثل `sum`)
df_dask_result = df_dask_numeric.sum().compute()
mem_during = max(mem_during, get_memory_usage_percent())  # تحديث أعلى استهلاك للذاكرة

end = time.time()
mem_after = get_memory_usage()  # قياس الذاكرة بعد التنفيذ

# ✅ إنشاء DataFrame باللغة الإنجليزية
results_df = pd.DataFrame(columns=["Operation Type", "Time Taken (Seconds)", "Peak Memory Usage (%)", "Memory Increase (MB)"])
results_df.loc[0] = [operation_type, round(end - start, 2), round(mem_during, 2), round(mem_after - mem_before, 2)]

# ✅ حفظ النتائج في ملف CSV دون حذف البيانات السابقة
results_csv_path = "results.csv"
try:
    existing_results = pd.read_csv(results_csv_path)
    results_df = pd.concat([existing_results, results_df], ignore_index=True)
except FileNotFoundError:
    pass  # إذا لم يكن هناك ملف سابق، يتم إنشاؤه مباشرة

results_df.to_csv(results_csv_path, index=False, encoding="utf-8", sep=",")

# ✅ عرض النتائج في جدول منسق
print("✅ Results saved in results.csv\n")
print(tabulate(results_df, headers="keys", tablefmt="grid"))

# ✅ تحرير الذاكرة بعد انتهاء المعالجة
del df_dask, df_dask_numeric, df_dask_result
gc.collect()


✅ Results saved in results.csv

+----+---------------------------+------------------------+-------------------------+------------------------+
|    | Operation Type            |   Time Taken (Seconds) |   Peak Memory Usage (%) |   Memory Increase (MB) |
+====+===========================+========================+=========================+========================+
|  0 | Pandas (chunksize=100000) |                  15.96 |                    12.7 |                 241.02 |
+----+---------------------------+------------------------+-------------------------+------------------------+
|  1 | Dask (direct read)        |                  12.65 |                    12.7 |                  32.87 |
+----+---------------------------+------------------------+-------------------------+------------------------+


218

In [ ]:
pip install memory_profiler

In [ ]:
import gzip

# ✅ تحديد مسار ملف CSV الأصلي والمضغوط
csv_file_path = "full_data.csv"
gzipped_csv_path = "full_data.gz"

# ✅ ضغط الملف إلى Gzip
with open(csv_file_path, "rb") as f_in:
    with gzip.open(gzipped_csv_path, "wb") as f_out:
        f_out.writelines(f_in)

print(f"✅ تم ضغط الملف '{csv_file_path}' إلى '{gzipped_csv_path}' بنجاح!")


✅ تم ضغط الملف 'full_data.csv' إلى 'full_data.gz' بنجاح!


In [ ]:
import os
print(os.listdir())  # يعرض جميع الملفات في المسار الحالي


['.config', 'full_data.csv', 'kaggle (3).json', 'medal-emnlp.zip', 'results.csv', 'full_data.gz', 'sample_data']


In [ ]:
!pip install psutil

# ***#########################################################***

In [ ]:
import dask.dataframe as dd
import time
import psutil
import os
import gc

# ✅ دالة قياس استهلاك الذاكرة
def get_memory_usage():
    return psutil.Process().memory_info().rss / (1024 * 1024)

# ✅ تحديد مسار ملف CSV الأصلي والمضغوط
gzipped_csv_path = "full_data.gz"
output_file_path = "describe_results.csv"  # مسار حفظ النتائج

# ✅ تحديد نوع العملية
operation_type = "Dask (Gzip Compressed)"
  compressed_file_path = os.path.join(gzipped_csv_path)
    start_time = time.time()
    start_memory = process.memory_info().rss / (1024 2)
    df = dd.read_csv(compressed_file_path, compression='gzip')

    print(f"🔹 Analyzing {file_name} (Compressed) using Dask:")
    print(df.describe().compute())
    end_time = time.time()
    end_memory = process.memory_info().rss / (1024  2)
    execution_time = end_time - start_time
    memory_used = end_memory - start_memory
    print(f"  - Execution Time: {execution_time:.2f} seconds")
    print(f"  - Memory Used: {memory_used:.2f} MB")
    print("-" * 80)

# ✅ طباعة النتائج مباشرة
print("\n✅ نتائج الأداء:")
print(f"🔹 نوع العملية: {operation_type}")
print(f"⏳ الزمن المستغرق: {round(end - start, 2)} ثانية")
print(f"📈 زيادة استهلاك الذاكرة: {round(mem_after - mem_before, 2)} ميجاب")


IndentationError: unexpected indent (<ipython-input-16-b894c572341f>, line 17)

# ***#########################################################***

**Dask and compers**

In [ ]:
import os
import pandas as pd
import gzip
import shutil

# تحديد مسار الملف
input_csv_path = 'full_data.csv'
chunk_size = 1000000  # عدد الصفوف في كل جزء
output_folder = 'chunks'  # مجلد لتخزين الأجزاء المضغوطة

# التأكد من وجود المجلد، وإنشاءه إذا لم يكن موجودًا
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# قراءة ملف CSV بأجزائه
for i, chunk in enumerate(pd.read_csv(input_csv_path, chunksize=chunk_size)):
    # تحديد مسار الجزء
    chunk_csv_path = os.path.join(output_folder, f"chunk_{i+1}.csv")

    # حفظ الجزء كملف CSV غير مضغوط
    chunk.to_csv(chunk_csv_path, index=False)

    # ضغط الجزء باستخدام gzip
    with open(chunk_csv_path, 'rb') as f_in:
        with gzip.open(f"{chunk_csv_path}.gz", 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)

    # حذف الملف غير المضغوط بعد الضغط
    os.remove(chunk_csv_path)

    print(f"تم إنشاء الجزء {i+1} وتم ضغطه.")

تم إنشاء الجزء 1 وتم ضغطه.


In [ ]:
import os
import time
import psutil
import dask.dataframe as dd
import gc
import pandas as pd
from tabulate import tabulate

# دالة لمعالجة الأجزاء المضغوطة باستخدام Dask
def process_chunks_with_dask(input_folder):
    # قياس استهلاك الذاكرة قبل التنفيذ
    process = psutil.Process(os.getpid())

    # قراءة جميع الملفات المضغوطة باستخدام Dask
    files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith('.csv.gz')]

    total_time = 0
    total_memory_increase = 0

    # معالجة كل ملف على حدة
    for file_path in files:
        # قياس استهلاك الذاكرة قبل المعالجة
        start_memory = process.memory_info().rss / (1024 * 1024)

        # تسجيل الوقت قبل المعالجة
        start_time = time.time()

        # قراءة الملف المضغوط باستخدام Dask
        ddf = dd.read_csv(file_path, compression='gzip')

        # معالجة البيانات باستخدام Dask
        print(f"معالجة الملف: {file_path}")
        result = ddf.describe().compute()
        print(result)

        # تسجيل الوقت بعد المعالجة
        end_time = time.time()

        # قياس استهلاك الذاكرة بعد المعالجة
        end_memory = process.memory_info().rss / (1024 * 1024)

        # حساب زيادة استهلاك الذاكرة لهذا الجزء
        memory_increase = end_memory - start_memory
        total_memory_increase += memory_increase

        # حساب الزمن المستغرق لهذا الجزء
        elapsed_time = end_time - start_time
        total_time += elapsed_time

        # مسح الذاكرة يدويًا
        del ddf
        gc.collect()  # جمع القمامة لتحرير الذاكرة

    # حفظ النتائج النهائية في ملف CSV
    operation_type = "Dask (processed chunks)"
    results_df = pd.DataFrame(columns=["Operation Type", "Time Taken (Seconds)", "Memory Increase (MB)"])
    results_df.loc[0] = [operation_type, round(total_time, 2), round(total_memory_increase, 2)]

    # حفظ النتائج في ملف CSV دون حذف البيانات السابقة
    results_csv_path = "results.csv"
    try:
        existing_results = pd.read_csv(results_csv_path)
        results_df = pd.concat([existing_results, results_df], ignore_index=True)
    except FileNotFoundError:
        pass  # إذا لم يكن هناك ملف سابق، يتم إنشاؤه مباشرة

    results_df.to_csv(results_csv_path, index=False, encoding="utf-8", sep=",")

    # عرض النتائج في جدول منسق
    print("✅ تم حفظ النتائج في results.csv")
    print(tabulate(results_df, headers="keys", tablefmt="grid"))

# استدعاء الدالة لمعالجة الأجزاء
process_chunks_with_dask('chunks')
